# Query Expansion with LLMs

Query expansion is a technique that improves search recall by generating alternative phrasings of the original query. A user searching for "how to fix memory issues" might also benefit from results matching "memory leak debugging", "RAM optimization", or "out of memory errors".

In the previous notebooks, we combined BM25 and vector search into a hybrid retrieval system. Now we'll improve the query side of the equation - using LLMs to generate alternative phrasings that improve recall without changing our document index.

In this notebook, we'll implement LLM-based query expansion using the OpenAI Responses API.

## Why Expand Queries?

Users often express the same information need in different ways:

| User Query | Might Also Want |
|------------|----------------|
| "fix slow computer" | "improve PC performance", "speed up laptop" |
| "Python error handling" | "try except blocks", "exception management" |
| "deploy to cloud" | "cloud deployment", "hosting on AWS/GCP" |

Query expansion bridges the vocabulary gap between how users phrase queries and how documents are written.

In [ ]:
%pip install -q openai pydantic

In [ ]:
import os
import json
from getpass import getpass
from openai import OpenAI

# API Key Setup
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

client = OpenAI()
MODEL = "gpt-5-mini"

## Basic Query Expansion

The simplest approach: ask the LLM to generate alternative queries.

In [ ]:
def expand_query(query: str, n_expansions: int = 3) -> list[str]:
    """Generate alternative phrasings of a search query."""
    
    prompt = f"""Generate {n_expansions} alternative search queries for the following query.
Each alternative should:
- Capture the same information need
- Use different keywords or phrasing
- Be concise (under 10 words)

Original query: {query}

Return ONLY a JSON array of strings, no explanation.
Example: ["alternative 1", "alternative 2", "alternative 3"]"""
    
    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    
    # Parse the JSON array
    try:
        expansions = json.loads(response.output_text)
        return expansions
    except json.JSONDecodeError:
        # Fallback: return original query
        return [query]

# Test it
original = "how to debug Python code"
expansions = expand_query(original)

print(f"Original: {original}")
print(f"Expansions:")
for i, exp in enumerate(expansions, 1):
    print(f"  {i}. {exp}")

## Structured Query Expansion

For production systems, we want more control over the expansion output. Let's use structured outputs.

In [ ]:
from pydantic import BaseModel, Field

class QueryExpansion(BaseModel):
    """Structured output for query expansion."""
    original: str = Field(description="The original query")
    synonyms: list[str] = Field(description="Queries using synonym terms")
    related_concepts: list[str] = Field(description="Queries for related concepts")
    specific: list[str] = Field(description="More specific versions of the query")
    general: list[str] = Field(description="More general versions of the query")


def expand_query_structured(query: str) -> QueryExpansion:
    """Generate structured query expansions."""
    
    prompt = f"""Expand the following search query into multiple alternative queries.

Original query: {query}

Generate:
- 2 queries using synonyms of the original terms
- 2 queries for related concepts
- 1 more specific query
- 1 more general query

Keep each query concise (under 10 words)."""

    # Get JSON schema from Pydantic model
    schema = QueryExpansion.model_json_schema()
    schema["additionalProperties"] = False
    
    response = client.responses.create(
        model=MODEL,
        input=prompt,
        text={
            "format": {
                "type": "json_schema",
                "name": "QueryExpansion",
                "strict": True,
                "schema": schema
            }
        }
    )
    
    return QueryExpansion.model_validate_json(response.output_text)


# Test structured expansion
expansion = expand_query_structured("machine learning best practices")

print(f"Original: {expansion.original}")
print(f"\nSynonyms:")
for q in expansion.synonyms:
    print(f"  - {q}")
print(f"\nRelated Concepts:")
for q in expansion.related_concepts:
    print(f"  - {q}")
print(f"\nMore Specific:")
for q in expansion.specific:
    print(f"  - {q}")
print(f"\nMore General:")
for q in expansion.general:
    print(f"  - {q}")

## Weighted Query Expansion

In production systems like QMD, the original query is weighted higher than expansions. This preserves exact match signals while still benefiting from expanded recall.

A common weighting scheme:
- Original query: weight = 2.0
- Expansions: weight = 1.0

**In plain English:** When we search with multiple query variations, we do not want the expanded queries to overpower the user's original words. Weighting the original at 2.0 and expansions at 1.0 means that if a document matches the original query, it gets twice the relevance boost compared to matching an expansion. This ensures the results the user specifically asked for stay at the top, while still surfacing relevant results that match alternative phrasings.

In [ ]:
from dataclasses import dataclass

@dataclass
class WeightedQuery:
    text: str
    weight: float

def expand_with_weights(query: str, original_weight: float = 2.0) -> list[WeightedQuery]:
    """Expand query and return weighted query list."""
    
    # Start with the original query at higher weight
    weighted_queries = [WeightedQuery(query, original_weight)]
    
    # Get expansions
    expansions = expand_query(query, n_expansions=2)
    
    # Add expansions at weight 1.0
    for exp in expansions:
        weighted_queries.append(WeightedQuery(exp, 1.0))
    
    return weighted_queries

# Test
weighted = expand_with_weights("Python async programming")
print("Weighted queries:")
for wq in weighted:
    print(f"  [{wq.weight:.1f}x] {wq.text}")

## Context-Aware Expansion

Expansions should be aware of the domain. "Python" in a programming context means the language, not the snake.

In [ ]:
def expand_query_with_context(query: str, context: str, n_expansions: int = 3) -> list[str]:
    """Generate context-aware query expansions."""
    
    prompt = f"""Generate {n_expansions} alternative search queries.

Context: {context}
Original query: {query}

The alternatives should:
- Be relevant to the given context
- Capture the same information need
- Use domain-appropriate terminology

Return ONLY a JSON array of strings."""
    
    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    
    try:
        return json.loads(response.output_text)
    except json.JSONDecodeError:
        return [query]

# Compare expansions with different contexts
query = "memory management"

contexts = [
    "Software engineering and programming",
    "Cognitive psychology and learning",
    "Computer hardware and systems"
]

print(f"Original query: '{query}'\n")

for ctx in contexts:
    print(f"Context: {ctx}")
    expansions = expand_query_with_context(query, ctx, n_expansions=2)
    for exp in expansions:
        print(f"  - {exp}")
    print()

## Caching Query Expansions

LLM calls are slow. Production systems cache expansions to avoid repeated API calls for the same queries.

In [ ]:
import hashlib
import time

class CachedQueryExpander:
    """Query expander with in-memory caching."""
    
    def __init__(self):
        self.cache = {}
        self.hits = 0
        self.misses = 0
    
    def _cache_key(self, query: str) -> str:
        """Generate cache key from normalized query."""
        normalized = query.lower().strip()
        return hashlib.md5(normalized.encode()).hexdigest()
    
    def expand(self, query: str, n_expansions: int = 3) -> list[str]:
        """Expand query with caching."""
        key = self._cache_key(query)
        
        if key in self.cache:
            self.hits += 1
            return self.cache[key]
        
        self.misses += 1
        expansions = expand_query(query, n_expansions)
        self.cache[key] = expansions
        return expansions
    
    def stats(self) -> dict:
        """Return cache statistics."""
        total = self.hits + self.misses
        hit_rate = self.hits / total if total > 0 else 0
        return {
            "hits": self.hits,
            "misses": self.misses,
            "hit_rate": f"{hit_rate:.1%}",
            "cache_size": len(self.cache)
        }

# Test caching
expander = CachedQueryExpander()

queries = [
    "Python debugging",
    "python debugging",  # Same query, different case - should hit cache
    "JavaScript frameworks",
    "Python debugging",  # Repeat - should hit cache
]

for q in queries:
    start = time.time()
    result = expander.expand(q)
    elapsed = time.time() - start
    print(f"'{q}' -> {len(result)} expansions ({elapsed*1000:.0f}ms)")

print(f"\nCache stats: {expander.stats()}")

## Measuring Expansion Quality

How do we know if our expansions are good? We need to check:
1. **Semantic similarity**: Expansions should be related to the original
2. **Lexical diversity**: Expansions should use different words
3. **Relevance preservation**: Expansions shouldn't drift too far from the original intent

In [ ]:
def evaluate_expansion(original: str, expansion: str) -> dict:
    """Have the LLM evaluate an expansion's quality."""
    
    prompt = f"""Evaluate this query expansion on a scale of 1-5 for each criterion.

Original: {original}
Expansion: {expansion}

Criteria:
- semantic_similarity: Does the expansion capture the same information need? (1=unrelated, 5=identical intent)
- lexical_diversity: Does it use different words? (1=same words, 5=completely different vocabulary)
- specificity: Is it appropriately specific? (1=too vague/specific, 5=well-calibrated)

Return ONLY a JSON object with these three keys and integer scores."""

    response = client.responses.create(
        model=MODEL,
        input=prompt
    )
    
    try:
        return json.loads(response.output_text)
    except:
        return {"error": "Failed to parse evaluation"}

# Evaluate some expansions
original = "Python error handling"
test_expansions = [
    "Python exception management",  # Good - different words, same concept
    "Python try except blocks",      # Good - more specific
    "JavaScript error handling",     # Bad - wrong language
    "coding",                         # Bad - too vague
]

print(f"Original: '{original}'\n")
for exp in test_expansions:
    scores = evaluate_expansion(original, exp)
    print(f"'{exp}'")
    for key, value in scores.items():
        print(f"  {key}: {value}")
    print()

---

## Exercises

### Exercise 1: Negative Expansion

Sometimes we want to exclude certain results. Implement a function that generates "negative queries" - things the user is NOT looking for.

Example:
- Query: "Python web frameworks"
- Negative: "Python snake", "Python Monty"

These can be used to filter out irrelevant results.

In [ ]:
def generate_negative_queries(query: str, context: str = None) -> list[str]:
    """Generate queries for things the user is NOT looking for."""
    # Your implementation here
    pass

### Exercise 2: Iterative Expansion

Implement "chain of thought" expansion where each expansion builds on the previous one, going from general to specific.

In [ ]:
def iterative_expansion(query: str, depth: int = 3) -> list[str]:
    """Generate increasingly specific queries through iteration."""
    # Your implementation here
    pass

### Exercise 3: Cross-Language Expansion

For multilingual corpora, generate expansions in multiple languages.

In [ ]:
def multilingual_expansion(query: str, languages: list[str]) -> dict[str, list[str]]:
    """Generate expansions in multiple languages."""
    # Your implementation here
    pass

---

## Summary

- Query expansion improves search recall by generating alternative phrasings
- LLMs can generate context-aware, semantically related expansions
- Original queries should be weighted higher than expansions to preserve precision
- Caching is essential for production systems to avoid repeated LLM calls
- Expansion quality can be measured via semantic similarity and lexical diversity

In the next notebook, we'll use query expansion as a core component of **RAG-Fusion** - generating multiple query variations, running parallel retrieval, and combining results using **Reciprocal Rank Fusion (RRF)**.